# Minimal first logistic-regression model

This is the small learning version of notebook 05. It contains only the essential path:

```text
load → time split → preprocessing pipeline → train → probability → evaluate
```

Weeks 72–91 remain locked. Calibration, thresholds, weekly stability, coefficient analysis, report files, and model saving remain in the professional notebook.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 1. Locate the data and choose the ten features

In [2]:
possible_roots = [Path.cwd(), Path.cwd().parent]
project_dir = next(
    path for path in possible_roots
    if (path / "data" / "parquet_files" / "train" / "train_base.parquet").exists()
)
train_dir = project_dir / "data" / "parquet_files" / "train"

features = [
    "credamount_770A",
    "annuity_780A",
    "maininc_215A",
    "applications30d_658L",
    "numactivecreds_622L",
    "totaldebt_9A",
    "avgdbddpdlast24m_3658932P",
    "avgdpdtolclosure24_3658938P",
    "clientscnt12m_3712952L",
    "price_1097A",
]

## 2. Load and join only the required columns

In [3]:
base_df = pd.read_parquet(
    train_dir / "train_base.parquet",
    columns=["case_id", "WEEK_NUM", "target"],
)

static_df = pd.concat(
    [
        pd.read_parquet(
            train_dir / f"train_static_0_{part}.parquet",
            columns=["case_id", *features],
        )
        for part in (0, 1)
    ],
    ignore_index=True,
)

df = base_df.merge(static_df, on="case_id", validate="one_to_one")
print(f"Loaded {len(df):,} applications")

Loaded 1,526,659 applications


## 3. Create the time split

- Training: weeks 0–51
- Validation: weeks 52–71
- Weeks 72–91 are not used here.

In [4]:
train_df = df.loc[df["WEEK_NUM"].between(0, 51)]
validation_df = df.loc[df["WEEK_NUM"].between(52, 71)]

X_train = train_df[features]
y_train = train_df["target"]
X_validation = validation_df[features]
y_validation = validation_df["target"]

print("Training rows:  ", f"{len(X_train):,}")
print("Validation rows:", f"{len(X_validation):,}")

Training rows:   1,065,921
Validation rows: 248,535


## 4. Create one pipeline

The pipeline learns all three stages from training data:

1. Replace missing values using training medians and add missing indicators.
2. Standardize values using training means and standard deviations.
3. Learn logistic-regression coefficients.

In [5]:
model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median", add_indicator=True),
    ),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=500)),
])

model

Pipeline(steps=[('imputer',
                 SimpleImputer(add_indicator=True, strategy='median')),
                ('scaler', StandardScaler()),
                ('logistic_regression', LogisticRegression(max_iter=500))])

## 5. Train using training rows only

In [6]:
model.fit(X_train, y_train)
print("Model training complete")

Model training complete


## 6. Predict validation probabilities

`[:, 1]` selects the probability of `target = 1`, which means default.

In [7]:
validation_probability = model.predict_proba(X_validation)[:, 1]

probability_examples = pd.DataFrame({
    "actual_target": y_validation.head(10).to_numpy(),
    "predicted_default_probability": validation_probability[:10],
})

probability_examples

,actual_target,predicted_default_probability
0,0,0.035105
1,0,0.024794
2,0,0.025080
3,0,0.025088
4,0,0.028360
5,0,0.024238
6,0,0.026513
7,0,0.028492
8,0,0.025080
9,0,0.027339


## 7. Evaluate the first model

- ROC AUC measures ranking ability; 0.50 is random ranking.
- Gini is `2 × AUC − 1`.
- Average precision focuses on the uncommon default class.

In [8]:
auc = roc_auc_score(y_validation, validation_probability)
gini = 2 * auc - 1
average_precision = average_precision_score(
    y_validation,
    validation_probability,
)

results = pd.Series({
    "Validation applications": len(y_validation),
    "Actual validation default rate": y_validation.mean(),
    "Mean predicted probability": validation_probability.mean(),
    "ROC AUC": auc,
    "Gini": gini,
    "Average precision": average_precision,
})

results

Validation applications           248535.000000
Actual validation default rate         0.041881
Mean predicted probability             0.031706
ROC AUC                                0.608473
Gini                                   0.216947
Average precision                      0.072298
dtype: float64

## Minimal workflow complete

Understand this notebook first. The larger professional notebook adds diagnostics around the same trained pipeline; it does not change the basic six-step process.